# 09 - Stage 11, part A: tables and figures from the frozen final run

**Purpose.** The analyses of the submitted report that need no training, computed from the saved
forecasts of the final run `20261114T083930Z_final` (tag `g1-frozen`): the benchmark with
intervals, the horizon, the bias and variance of the error, the production quintiles, the error by
season and local hour, the encoder length of the TFT, PatchTST against XGB, and the comparison
with the literature on one normalisation.

**Rules.**
- Nothing is trained and the step `test` is not run again. The test rows are read only to score the
  frozen forecasts; each read is in `outputs/07_evaluation/test_access_log.csv`.
- Every test number is placed beside the validation number of the same view.
- No winner is declared (D-37). Four of the five neural models have one seed.

**Documented in** `docs/stage11_plan.md`, OP-66, run journal R11-01, errata C-78 to C-84 (M45-M50)
and X100-X106, tests T-169 to T-177, and the figures S11-01 to S11-17.

## 1. What the module checked before it wrote anything

In [1]:
import json

import pandas as pd

from solarpv.config.paths import REPO_ROOT

A = REPO_ROOT / "outputs" / "11_analyses"
man = json.loads((A / "manifest.json").read_text(encoding="utf-8"))
pd.Series({"run": man["run"], "tables": len(man["rows"]),
           "input files": len(man["inputs_sha256"]),
           "max difference of nRMSE vs the run's report (pp)":
               man["max_abs_diff_vs_report_nRMSE_kW_pct"],
           "max relative error of MSE = ME^2 + Var(e)":
               man["max_rel_identity_error_bias2_plus_variance"]})

run                                                 outputs/runs/20261114T083930Z_final
tables                                                                               14
input files                                                                          19
max difference of nRMSE vs the run's report (pp)                                    0.0
max relative error of MSE = ME^2 + Var(e)                                           0.0
dtype: object

**What it shows.** The module reads 19 prediction files, each
checked against the SHA-256 of its step record, and scores them in kW against the actual AC.

**What it means.** The nRMSE of every model, seed, route, horizon and split agrees with the run's
own report tables to 5.0e-07 (the tables are written with
six decimals), so the new tables use the same errors as the acceptance and the test step (X100).
The decomposition MSE = ME^2 + Var(e) holds to 4e-16.

## 2. The benchmark in kW, with intervals (S-23)

In [2]:
ci = pd.read_csv(A / "benchmark_ci_kw.csv")
g = ci.groupby(["model", "target", "horizon", "split"])[["nRMSE_kW_pct", "ci_lo", "ci_hi"]].mean()
t = g.xs(("indirect", 1), level=("target", "horizon")).round(2)
t.unstack("split")

nRMSE_kW_pct        ci_lo        ci_hi       
split               test   val   test   val   test    val
model                                                    
clearsky_k         12.85  8.96  10.93  7.66  14.53  10.34
climatology        12.37  8.06  10.59  6.78  13.99   9.43
lstm_sota          11.61  6.99   9.66  5.87  13.63   8.17
patchtst           13.44  8.77  11.36  7.31  15.29  10.35
persistence        15.71  9.68  13.67  8.32  17.55  11.12
tft                11.39  6.83   9.44  5.75  13.37   8.03
tft_168            11.21  6.82   9.25  5.71  13.20   7.97
tft_96             11.45  6.74   9.46  5.67  13.43   7.91
xgb_sota           11.28  6.63   9.32  5.56  13.27   7.75

**What it shows.** The nRMSE in kW (percent of 3,449 kW) of the indirect route at D+1, with a
95 % circular block bootstrap interval, mean over seeds, validation and test, for the six models and
the three references (`S11-01` draws all six cells).

**What it means.** The half-width of the interval of one score is 1.17 pp on validation
and 1.93 pp on test (mean over the six models). These intervals are not paired, so their
overlap is not a test of a difference; the paired test is S10-04.

## 3. The horizon (S-26)

In [3]:
hr = pd.read_csv(A / "horizon_robustness.csv")
hr[hr["target"] == "indirect"].round(2).set_index(["model", "split"])[
    ["D1", "D2", "D3", "D3_minus_D1_pp"]]

D1     D2     D3  D3_minus_D1_pp
model       split                                     
clearsky_k  test   12.85  12.85  12.85            0.00
            val     8.96   8.96   8.96            0.00
climatology test   12.37  12.37  12.37            0.00
            val     8.06   8.06   8.06            0.00
lstm_sota   test   11.61  11.85  11.87            0.27
            val     6.99   7.17   6.84           -0.15
patchtst    test   13.44  13.63  13.56            0.12
            val     8.77   8.90   8.86            0.09
persistence test   15.71  17.46  17.57            1.86
            val     9.68  10.84  10.76            1.08
tft         test   11.39  11.59  11.77            0.38
            val     6.83   7.10   6.81           -0.02
tft_168     test   11.21  11.33  11.52            0.31
            val     6.82   7.04   6.60           -0.22
tft_96      test   11.45  11.60  11.64            0.20
            val     6.74   6.94   6.57           -0.17
xgb_sota    test   11.28  11.37  11.46            0.18
            val     6.63   6.88   6.56           -0.07

**What it shows.** The seed-mean nRMSE of every model and reference at D+1, D+2 and D+3, and
the change from D+1 to D+3 (the direct route is in the file).

**What it means.** On validation the error at D+3 is lower than at D+1 in 10 of the 12 pairs of
a model and a route; on test it is higher in 12 of 12. The sign of the horizon effect reverses
between the two periods, and its size is at most
0.38 pp: the order of the horizons is a property of the
period (R6b-07, R7-05), not of the horizon (`S11-02`).

## 4. Bias² and variance, and where the bias comes from (S-39, L-15)

In [4]:
dec = pd.read_csv(A / "decomposition.csv")
d = dec[(dec["split"] == "test") & (dec["horizon"] == 1) & (dec["target"] == "indirect")]
d.pivot_table(index="model", columns="view", values="me_kw").round(1).join(
    d[d["view"] == "all"].groupby("model")[["bias2_share", "variance_ratio"]].mean().round(3))

,all,normal_operation,bias2_share,variance_ratio
model,,,,
clearsky_k,63.8,2.1,0.021,0.916
climatology,36.6,-22.7,0.007,0.822
lstm_sota,70.7,6.9,0.033,0.922
patchtst,161.8,100.4,0.122,0.938
persistence,-0.1,-49.0,0.000,1.000
tft,71.7,7.4,0.033,0.913
tft_168,50.8,-13.0,0.017,0.913
tft_96,77.4,13.2,0.038,0.894
xgb_sota,55.4,-7.8,0.020,0.892


**What it shows.** For the indirect route at D+1 on test: the mean error in kW on all daytime
hours and on the hours of normal operation (no curtailment, stop or derate; D-22), the bias² share of the
MSE and the variance ratio V(f)/V(y) (L-15).

**What it means.** Bias² is at most 4.6% of the squared error on validation and
6.0% on test for the five models other than PatchTST. Every model has a positive mean error on
all hours (+52 to +125 kW on test), and for XGB, BiLSTM and the three TFTs
it is -11 to
+12 kW in normal operation: the positive bias of
the neural forecasts (a G-1 list point, D-37) is made on the operation-limited hours. PatchTST keeps
+64 kW. The variance between the three seeds is at most 2.8% of the
squared error (`S11-11`). The submitted section 12.3 had no definition of "variance"; here it is the
variance of the error, and the seed split is a separate table (M46).

## 5. Production quintiles and the control for binning by the outcome (S-40, M45)

In [5]:
qu = pd.read_csv(A / "quintiles.csv")
qc = pd.read_csv(A / "quintile_control.csv")
q = qu[(qu["model"] == "xgb_sota") & (qu["seed"] == 42) & (qu["target"] == "direct")
       & (qu["horizon"] == 1) & (qu["split"] == "test")]
t = q.pivot(index="bin", columns="binned_by", values="me_kw").round(0)
t["calibrated control, observed bins"] = qc[(qc["target"] == "direct") & (qc["horizon"] == 1)
    & (qc["split"] == "test")].set_index("bin")["me_calibrated_kw"].round(0)
t["op-limited share, observed bins"] = q[q["binned_by"] == "observed"].set_index("bin")[
    "op_limited_share"].round(3)
t

binned_by,forecast,observed,"calibrated control, observed bins","op-limited share, observed bins"
bin,,,,
1,4.0,263.0,263.0,0.140
2,-9.0,101.0,105.0,0.083
3,24.0,18.0,1.0,0.061
4,87.0,-46.0,-92.0,0.031
5,162.0,-98.0,-170.0,0.037


**What it shows.** The mean error of the XGB (seed 42, direct route, D+1, test) in five bins of
equal validation count, binned by the observed power and by the forecast; a control (the same forecast
made unbiased on validation by an isotonic regression) binned by the observed power; and the share of
operation-limited hours in each observed bin.

**What it means.** Binned by the observed power the lowest bin has +263 kW and the
highest -98 kW, the pattern that the submitted report read as inverter downtime and
as underestimation of the peak. Binned by the forecast it is +4 and
+162 kW. The control forecast, unbiased by construction, gives
+263 and -170 kW in the observed bins, so
the pattern comes with binning by the outcome (M45). 14% of the hours of
the lowest observed bin are operation-limited (`S11-12`).

## 6. Error by season and by local hour (S-29, S-30, L-20)

In [6]:
sea = pd.read_csv(A / "seasonal_kw.csv")
s = sea[(sea["target"] == "indirect") & (sea["horizon"] == 1)
        & sea["model"].isin(["xgb_sota", "climatology"])]
s.pivot_table(index=["model", "season"], columns="split", values="nRMSE_kW_pct").round(2).join(
    s[s["model"] == "xgb_sota"].set_index(["model", "season", "split"])["months"]
    .unstack("split").add_prefix("months "), how="left")

split                test    val       months test        months val
model       season                                                  
climatology Fall    13.68   4.22               NaN               NaN
            Spring  10.98  10.29               NaN               NaN
            Summer    NaN   2.86               NaN               NaN
            Winter  13.06  13.86               NaN               NaN
xgb_sota    Fall    14.19   3.70  2024-10..2024-11  2024-09..2024-09
            Spring   8.89   8.95  2025-03..2025-05  2024-03..2024-05
            Summer    NaN   2.01               NaN  2024-06..2024-08
            Winter  11.67  10.05  2024-12..2025-02  2024-02..2024-02

**What it shows.** The nRMSE in kW by season for XGB and climatology, with the local months
that each split holds in that season.

**What it means.** The test split has no summer. Its autumn is 2024-10..2024-11 and the
validation autumn is 2024-09..2024-09, so a season is not the same weather in the two periods
(X104). XGB's error in the autumn is 3.7 % on validation and
14.2 % on test, and the rise of the test period is concentrated there
(`S11-04`).

In [7]:
hs = pd.read_csv(A / "hour_season.csv")
h = hs[(hs["split"] == "test") & (hs["target"] == "indirect") & (hs["horizon"] == 1)
       & (hs["model"] == "xgb_sota") & (hs["season"] == "Fall") & (hs["n"] >= 20)]
h.pivot(index="hour", columns="view", values="nRMSE_kW_pct").round(1).join(
    h[h["view"] == "all"].set_index("hour")["n"])

,all,normal_operation,n
hour,,,
7,2.7,2.7,61
8,5.7,4.9,61
9,16.2,5.3,61
10,21.8,5.3,61
11,22.9,4.4,61
12,19.0,3.9,61
13,14.7,5.1,61
14,10.9,4.3,61
15,5.8,2.4,61


**What it shows.** XGB on test in the autumn, by local hour: the nRMSE on all hours and in normal
operation, and the number of hours.

**What it means.** At 11:00 the error is 22.9 % on all hours and
4.4 % in normal operation: the autumn peak of the diurnal
profile (`S11-05`) is mostly the operation-limited hours, which cluster around midday; in normal operation
the profile is flat (`S11-17`). The submitted "afternoon shoulder" is not seen (M47).

## 7. Two paired comparisons (S-34, S-35)

In [8]:
enc = pd.read_csv(A / "encoder_length.csv")
pt = pd.read_csv(A / "patchtst_gap.csv")
display(enc[["split", "target", "horizon", "diff_a_minus_b_pp", "p_value_hln", "pct_of_mde",
             "significant"]].round(3))
pt[["split", "target", "horizon", "model_b", "diff_a_minus_b_pp", "p_value_hln", "significant"]
   ].round(3)

,split,target,horizon,diff_a_minus_b_pp,p_value_hln,pct_of_mde,significant
0,val,indirect,1,0.078,0.426,20.978,False
1,val,indirect,2,0.095,0.434,24.082,False
2,val,indirect,3,0.022,0.847,5.673,False
3,val,direct,1,0.099,0.189,44.211,False
4,val,direct,2,0.086,0.275,36.044,False
5,val,direct,3,-0.022,0.785,8.935,False
6,test,indirect,1,-0.240,0.034,81.201,False
7,test,indirect,2,-0.272,0.026,128.719,False
8,test,indirect,3,-0.122,0.330,32.967,False
9,test,direct,1,0.286,0.013,70.194,False


,split,target,horizon,model_b,diff_a_minus_b_pp,p_value_hln,significant
0,val,indirect,1,xgb_sota,2.122,0.000,True
1,val,indirect,1,climatology,0.704,0.000,True
2,val,indirect,2,xgb_sota,2.016,0.001,True
3,val,indirect,2,climatology,0.842,0.002,True
4,val,indirect,3,xgb_sota,2.314,0.001,True
5,val,indirect,3,climatology,0.797,0.004,True
6,val,direct,1,xgb_sota,1.461,0.002,True
7,val,direct,1,climatology,0.049,0.681,False
8,val,direct,2,xgb_sota,1.471,0.003,True
9,val,direct,2,climatology,0.225,0.137,False


**What it shows.** TFT-168 minus TFT-96 (a positive difference: the longer encoder is worse) and
PatchTST minus XGB and minus climatology, in kW, with the paired Diebold-Mariano test (HLN, Bonferroni
over the six cells of a split).

**What it means.** The encoder length: on validation the difference is
-0.02 to
+0.10 pp with no significant cell; on test it is
negative for the indirect route and positive for the direct route, and 1 of
12 cells is significant. A consistent advantage of the shorter window, as the submitted report states, is
not found (M48, `S11-09`). PatchTST is worse than XGB by
1.36 to 2.31 pp, significantly in every
cell, and worse than climatology by +0.05 to
+1.26 pp (significant in 8 of 12): the
submitted bound on the value of feature engineering is not stated (M49, `S11-14`). Both comparisons have
one seed per model.

## 8. The literature, on one normalisation (L-20, L-16)

In [9]:
lit = pd.read_csv(A / "literature_alignment.csv")
x = lit[lit["model"] != "literature"].groupby(["model", "split"])["nRMSE_mean_pct"].mean().unstack()
display(x.round(1))
lit[lit["model"] == "literature"][["source", "nRMSE_mean_pct"]]

split,test,val
model,,
clearsky_k,29.9,20.4
climatology,28.9,18.4
lstm_sota,26.8,15.5
patchtst,30.4,19.2
persistence,36.5,22.1
tft,26.5,15.6
tft_168,26.3,15.5
tft_96,26.2,15.3
xgb_sota,26.2,15.1


,source,nRMSE_mean_pct
36,"C1 (2-day persistence, mean of 7 Hungarian pla...",81.6
37,C1 (physical model chain),48.7
38,"C1 (best tuned models (RNN, Pyraformer))",42.7
39,"C1 (worst tuned models (XGBoost, LightGBM))",43.5
40,"C2 (best workflow, ensemble NWP with power bia...",37.4


**What it shows.** Our D+1 nRMSE normalised by the mean observed daytime power (the
normalisation of the papers of `docs/literature_review.md`), and the papers' numbers.

**What it means.** Our six models are 21
to 25 % (validation and test
together by model), the papers 37.4 to 48.7 % for tuned models. The
numbers are not comparable one to one: a hyper-arid site where most hours are clear, GFS instead of a 9 km
ECMWF run, hourly instead of 15-minute data, one plant instead of a mean over seven, and an eight-month test
period. No ranking against the papers is made (M50, `S11-15`).

## 9. What part A did not do

- **Dust days (L-20).** The project holds no aerosol record; the definition of a dust day is Q-29 for
  the user.
- **`reproduce_paper_results.py --xgb-only` and `--skip-train` (S-45).** The script is outside the D-36
  list; Q-30 for the user.
- **The report text** (S-01, S-05, S-42 to S-44) and the final table of old against new values: part B.